# Project 4 — Tokenizer Explorer

Explored and compared tokenization behavior across three major 
transformer model families: BERT, GPT2, and RoBERTa.

## Key Findings

### Architecture Differences
| Model | Type | Special Tokens | Case |
|---|---|---|---|
| BERT | Encoder | [CLS], [SEP], [MASK] | Lowercase |
| GPT2 | Decoder | <endoftext> only | Preserved |
| RoBERTa | Encoder | <s>, </s>, <mask> | Preserved |

### Tokenization Behavior
- BERT uses ## to mark subword continuation
- GPT2/RoBERTa use Ġ to mark spaces before tokens
- Hindi text costs 6x more tokens than English in GPT2/RoBERTa
- BERT collapses non-English text to [UNK] — loses all meaning
- Special characters cost 2x more tokens in BERT vs GPT2

### Production Implications
- Use multilingual models (xlm-roberta) for non-English text
- GPT2/RoBERTa more efficient for special characters and social media
- Domain-specific terms (Kubernetes, COVID) get split — fine-tuning helps
- Token count directly affects API cost and context window usage

## Tech Stack
- Models: bert-base-uncased, gpt2, roberta-base
- Library: HuggingFace Transformers

In [1]:
!pip install transformers -q

In [2]:
from transformers import AutoTokenizer

tokenizers = {
    "BERT":     AutoTokenizer.from_pretrained("bert-base-uncased"),
    "GPT2":     AutoTokenizer.from_pretrained("gpt2"),
    "RoBERTa":  AutoTokenizer.from_pretrained("roberta-base"),
}

print("All tokenizers loaded successfully")
for name, tok in tokenizers.items():
    print(f"{name}: vocab size = {tok.vocab_size:,}")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

All tokenizers loaded successfully
BERT: vocab size = 30,522
GPT2: vocab size = 50,257
RoBERTa: vocab size = 50,265


In [3]:
sentences = [
    "Hello world",
    "Kubernetes orchestrates containerized applications",
    "supercalifragilistic",
    "!!??##@@",
    "I love distributed systems",
    "ChatGPT is an LLM built by OpenAI",
    "मैं ठीक हूँ",
    "machine learning is fun!!!",
]

for sentence in sentences:
    print(f"\nSentence: '{sentence}'")
    print("-" * 60)
    for name, tok in tokenizers.items():
        tokens = tok.tokenize(sentence)
        print(f"  {name:<10} ({len(tokens)} tokens): {tokens}")


Sentence: 'Hello world'
------------------------------------------------------------
  BERT       (2 tokens): ['hello', 'world']
  GPT2       (2 tokens): ['Hello', 'Ġworld']
  RoBERTa    (2 tokens): ['Hello', 'Ġworld']

Sentence: 'Kubernetes orchestrates containerized applications'
------------------------------------------------------------
  BERT       (9 tokens): ['ku', '##ber', '##net', '##es', 'orchestra', '##tes', 'container', '##ized', 'applications']
  GPT2       (9 tokens): ['K', 'uber', 'net', 'es', 'Ġorchestr', 'ates', 'Ġcontainer', 'ized', 'Ġapplications']
  RoBERTa    (9 tokens): ['K', 'uber', 'net', 'es', 'Ġorchestr', 'ates', 'Ġcontainer', 'ized', 'Ġapplications']

Sentence: 'supercalifragilistic'
------------------------------------------------------------
  BERT       (6 tokens): ['super', '##cal', '##if', '##rag', '##ilis', '##tic']
  GPT2       (6 tokens): ['super', 'cal', 'if', 'rag', 'il', 'istic']
  RoBERTa    (6 tokens): ['super', 'cal', 'if', 'rag', 'il', 'istic

In [4]:
import pandas as pd

test_sentences = [
    "Hello",
    "Kubernetes orchestrates containerized applications across clusters",
    "Fine-tuning adapts pretrained models to specific downstream tasks",
    "supercalifragilistic",
    "मैं ठीक हूँ",
    "!!??##@@***$$$",
    "sabinar",
    "ChatGPT revolutionized natural language processing in 2022",
]

rows = []
for sentence in test_sentences:
    row = {"sentence": sentence[:45]}
    for name, tok in tokenizers.items():
        row[name] = len(tok.tokenize(sentence))
    rows.append(row)

df = pd.DataFrame(rows)
print(df.to_string(index=False))

                                     sentence  BERT  GPT2  RoBERTa
                                        Hello     1     1        1
Kubernetes orchestrates containerized applica    11    11       11
Fine-tuning adapts pretrained models to speci    13    13       13
                         supercalifragilistic     6     6        6
                                  मैं ठीक हूँ     3    18       18
                               !!??##@@***$$$    14     7        7
                                      sabinar     3     3        3
ChatGPT revolutionized natural language proce    11    10       10


In [5]:
print("SPECIAL TOKENS PER MODEL")
print("=" * 60)

sample = "I love distributed systems"

for name, tok in tokenizers.items():
    plain = tok.tokenize(sample)
    with_special = tok.encode(sample, add_special_tokens=True)
    decoded = tok.decode(with_special)
    
    print(f"\n{name}:")
    print(f"  Plain tokens:     {plain}")
    print(f"  With special IDs: {with_special}")
    print(f"  Decoded back:     {decoded}")
    print(f"  Special tokens:   {tok.all_special_tokens}")

SPECIAL TOKENS PER MODEL

BERT:
  Plain tokens:     ['i', 'love', 'distributed', 'systems']
  With special IDs: [101, 1045, 2293, 5500, 3001, 102]
  Decoded back:     [CLS] i love distributed systems [SEP]
  Special tokens:   ['[UNK]', '[SEP]', '[PAD]', '[CLS]', '[MASK]']

GPT2:
  Plain tokens:     ['I', 'Ġlove', 'Ġdistributed', 'Ġsystems']
  With special IDs: [40, 1842, 9387, 3341]
  Decoded back:     I love distributed systems
  Special tokens:   ['<|endoftext|>']

RoBERTa:
  Plain tokens:     ['I', 'Ġlove', 'Ġdistributed', 'Ġsystems']
  With special IDs: [0, 100, 657, 7664, 1743, 2]
  Decoded back:     <s>I love distributed systems</s>
  Special tokens:   ['<s>', '</s>', '<unk>', '<pad>', '<mask>']


In [7]:
print("HOW EACH MODEL HANDLES UNKNOWN/RARE WORDS")
print("=" * 50)

rare_words = [
    "transformerization",
    "Kubernetes",
    "COVID-19",
    "GPT-4",
    "unfinetunable",
    "hallucination",
    "agentic",
]

for word in rare_words:
    print(f"\nWord: '{word}'")
    for name, tok in tokenizers.items():
        tokens = tok.tokenize(word)
        print(f"  {name:<10} ({len(tokens)} pieces): {tokens}")

HOW EACH MODEL HANDLES UNKNOWN/RARE WORDS

Word: 'transformerization'
  BERT       (3 pieces): ['transform', '##eri', '##zation']
  GPT2       (3 pieces): ['trans', 'former', 'ization']
  RoBERTa    (3 pieces): ['trans', 'former', 'ization']

Word: 'Kubernetes'
  BERT       (4 pieces): ['ku', '##ber', '##net', '##es']
  GPT2       (4 pieces): ['K', 'uber', 'net', 'es']
  RoBERTa    (4 pieces): ['K', 'uber', 'net', 'es']

Word: 'COVID-19'
  BERT       (4 pieces): ['co', '##vid', '-', '19']
  GPT2       (4 pieces): ['CO', 'VID', '-', '19']
  RoBERTa    (4 pieces): ['CO', 'VID', '-', '19']

Word: 'GPT-4'
  BERT       (4 pieces): ['gp', '##t', '-', '4']
  GPT2       (4 pieces): ['G', 'PT', '-', '4']
  RoBERTa    (4 pieces): ['G', 'PT', '-', '4']

Word: 'unfinetunable'
  BERT       (4 pieces): ['un', '##fine', '##tu', '##nable']
  GPT2       (5 pieces): ['un', 'fin', 'et', 'un', 'able']
  RoBERTa    (5 pieces): ['un', 'fin', 'et', 'un', 'able']

Word: 'hallucination'
  BERT       (3 pieces)